In [4]:
import pandas as pd

# ============================================================
# 1. LOAD ORIGINAL DATASET FROM RAW
# ============================================================

raw_path = "../data/raw/LI-Small_Trans.csv"

trans_df = pd.read_csv(
    raw_path,
    usecols=["Timestamp"]
)

print("Raw dataset loaded successfully.")
print("Total timestamps:", len(trans_df))


# ============================================================
# 2. SHOW EXAMPLES FROM RAW DATA
# ============================================================

print("\nFirst 10 raw timestamps:")
print(trans_df["Timestamp"].head(10).to_string(index=False))


# ============================================================
# 3. CHECK EXACT EXPECTED FORMAT
# ============================================================

expected_format = "%Y/%m/%d %H:%M"

parsed_timestamp = pd.to_datetime(
    trans_df["Timestamp"],
    format=expected_format,
    errors="coerce"
)


# ============================================================
# 4. CHECK INVALID TIMESTAMPS
# ============================================================

invalid_mask = parsed_timestamp.isna()

invalid_count = invalid_mask.sum()
valid_count = (~invalid_mask).sum()

print("\nTimestamp validation:")
print("Valid timestamps  :", valid_count)
print("Invalid timestamps:", invalid_count)


# ============================================================
# 5. SHOW INVALID VALUES IF ANY
# ============================================================

if invalid_count > 0:

    print("\nINVALID TIMESTAMP EXAMPLES:")

    print(
        trans_df.loc[invalid_mask, "Timestamp"]
        .head(20)
        .to_string(index=False)
    )

else:

    print("\nALL timestamps match the exact format:")
    print("YYYY/MM/DD HH:MM")


# ============================================================
# 6. CHECK THE DATE ORDER
# ============================================================

print("\nDate interpretation:")
print("Year  :", parsed_timestamp.dt.year.min(), "to", parsed_timestamp.dt.year.max())
print("Month :", parsed_timestamp.dt.month.min(), "to", parsed_timestamp.dt.month.max())
print("Day   :", parsed_timestamp.dt.day.min(), "to", parsed_timestamp.dt.day.max())


# ============================================================
# 7. SHOW PARSED EXAMPLES
# ============================================================

print("\nRaw → Parsed examples:")

check_df = pd.DataFrame({
    "Raw Timestamp": trans_df["Timestamp"].head(10),
    "Parsed Timestamp": parsed_timestamp.head(10)
})

print(check_df.to_string(index=False))


# ============================================================
# 8. FINAL RESULT
# ============================================================

if invalid_count == 0:

    print("\n" + "=" * 60)
    print("TIMESTAMP CHECK PASSED")
    print("=" * 60)

    print("Format confirmed: YYYY/MM/DD HH:MM")
    print("Python/Pandas format:", expected_format)
    print("Example:", trans_df["Timestamp"].iloc[0])

else:

    print("\n" + "=" * 60)
    print("TIMESTAMP CHECK FAILED")
    print("=" * 60)

    print("Some timestamps do not match the expected format.")
    print("DO NOT generate temporal features until they are checked.")

Raw dataset loaded successfully.
Total timestamps: 6924049

First 10 raw timestamps:
2022/09/01 00:08
2022/09/01 00:21
2022/09/01 00:00
2022/09/01 00:16
2022/09/01 00:00
2022/09/01 00:24
2022/09/01 00:17
2022/09/01 00:07
2022/09/01 00:28
2022/09/01 00:22

Timestamp validation:
Valid timestamps  : 6924049
Invalid timestamps: 0

ALL timestamps match the exact format:
YYYY/MM/DD HH:MM

Date interpretation:
Year  : 2022 to 2022
Month : 9 to 9
Day   : 1 to 17

Raw → Parsed examples:
   Raw Timestamp    Parsed Timestamp
2022/09/01 00:08 2022-09-01 00:08:00
2022/09/01 00:21 2022-09-01 00:21:00
2022/09/01 00:00 2022-09-01 00:00:00
2022/09/01 00:16 2022-09-01 00:16:00
2022/09/01 00:00 2022-09-01 00:00:00
2022/09/01 00:24 2022-09-01 00:24:00
2022/09/01 00:17 2022-09-01 00:17:00
2022/09/01 00:07 2022-09-01 00:07:00
2022/09/01 00:28 2022-09-01 00:28:00
2022/09/01 00:22 2022-09-01 00:22:00

TIMESTAMP CHECK PASSED
Format confirmed: YYYY/MM/DD HH:MM
Python/Pandas format: %Y/%m/%d %H:%M
Example: 2022/

In [5]:
import os
import gc
import time

import numpy as np
import pandas as pd

BASE_PATH = ".."
RAW_PATH = os.path.join(BASE_PATH, "data", "raw")
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")
os.makedirs(PROCESSED_PATH, exist_ok=True)

TRANS_FILE = os.path.join(RAW_PATH, "LI-Small_Trans.csv")
ACCOUNTS_FILE = os.path.join(RAW_PATH, "LI-Small_accounts.csv")   # optional validation only
OUTPUT_FILE = os.path.join(PROCESSED_PATH, "temporal_features.csv")

TIMESTAMP_FORMAT = "%Y/%m/%d %H:%M"      # e.g. 2022/09/01 00:08

# Window lengths in seconds
WINDOW_1H = 60 * 60
WINDOW_24H = 24 * 60 * 60
WINDOW_7D = 7 * 24 * 60 * 60

FINAL_COLUMNS = [
    "Transaction ID",
    "Sender Account ID",
    "Receiver Account ID",
    "Transactions Last 1 Hour",
    "Transactions Last 24 Hours",
    "Transactions Last 7 Days",
    "Amount Last 24 Hours",
    "Time Since Previous Transaction",
]

print("Transactions file:", os.path.abspath(TRANS_FILE))
print("Output file      :", os.path.abspath(OUTPUT_FILE))

Transactions file: e:\AML-GNN-Project\data\raw\LI-Small_Trans.csv
Output file      : e:\AML-GNN-Project\data\processed\temporal_features.csv


In [6]:
t_start = time.time()

USE_COLS = ["Timestamp", "From Bank", "Account", "To Bank", "Account.1", "Amount Paid"]

df = pd.read_csv(
    TRANS_FILE,
    usecols=USE_COLS,
    dtype={
        "Timestamp": "string",
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string",
        "Amount Paid": "float64",
    },
)
df = df[USE_COLS]   # fixed column order

N_RAW = len(df)
print(f"Rows loaded : {N_RAW:,}")
print(f"Columns     : {list(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
print(f"Load time   : {time.time() - t_start:,.1f} s")
df.head()

Rows loaded : 6,924,049
Columns     : ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Paid']
Memory usage: 1,973.5 MB
Load time   : 17.4 s


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Paid
0,2022/09/01 00:08,011,8000ECA90,011,8000ECA90,3195403.00
1,2022/09/01 00:21,03402,80021DAD0,03402,80021DAD0,1858.96
2,2022/09/01 00:00,011,8000ECA90,001120,8006AA910,592571.00
3,2022/09/01 00:16,03814,8006AD080,03814,8006AD080,12.32
4,2022/09/01 00:00,020,8006AD530,020,8006AD530,2941.56


In [7]:
# Transaction ID = original raw-file order (1..N). Created BEFORE any sorting.
df.insert(0, "Transaction ID", np.arange(1, N_RAW + 1, dtype=np.int64))

# Missing components would produce broken IDs, so check first
for col in ["From Bank", "Account", "To Bank", "Account.1"]:
    n_missing = int(df[col].isna().sum())
    if n_missing:
        raise ValueError(f"Column '{col}' has {n_missing:,} missing values; cannot build Account IDs.")

# Account ID = Bank + "_" + Account Number
df["Sender Account ID"] = df["From Bank"] + "_" + df["Account"]
df["Receiver Account ID"] = df["To Bank"] + "_" + df["Account.1"]

# Free the raw component columns
df.drop(columns=["From Bank", "Account", "To Bank", "Account.1"], inplace=True)

# Category dtype: each distinct ID string is stored once, and gives integer codes for fast sorting
df["Sender Account ID"] = df["Sender Account ID"].astype("category")
df["Receiver Account ID"] = df["Receiver Account ID"].astype("category")
gc.collect()

# Verify Transaction ID
assert df["Transaction ID"].is_unique, "Transaction ID is not unique"
assert df["Transaction ID"].iloc[0] == 1 and df["Transaction ID"].iloc[-1] == N_RAW
assert (np.diff(df["Transaction ID"].to_numpy()) == 1).all(), "Transaction ID is not 1..N"

print(f"Transaction ID range : 1 -> {N_RAW:,} (unique, consecutive)")
print(f"Unique sender accounts  : {df['Sender Account ID'].nunique():,}")
print(f"Unique receiver accounts: {df['Receiver Account ID'].nunique():,}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
df.head()

Transaction ID range : 1 -> 6,924,049 (unique, consecutive)
Unique sender accounts  : 681,283
Unique receiver accounts: 576,178
Memory usage: 666.1 MB


,Transaction ID,Timestamp,Amount Paid,Sender Account ID,Receiver Account ID
0,1,2022/09/01 00:08,3195403.00,011_8000ECA90,011_8000ECA90
1,2,2022/09/01 00:21,1858.96,03402_80021DAD0,03402_80021DAD0
2,3,2022/09/01 00:00,592571.00,011_8000ECA90,001120_8006AA910
3,4,2022/09/01 00:16,12.32,03814_8006AD080,03814_8006AD080
4,5,2022/09/01 00:00,2941.56,020_8006AD530,020_8006AD530


In [8]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format=TIMESTAMP_FORMAT, errors="coerce")

invalid_mask = df["Timestamp"].isna()
N_INVALID = int(invalid_mask.sum())
print(f"Invalid timestamps: {N_INVALID:,}")

# A very high invalid count means the format is wrong, not that the data is bad
if N_INVALID > 0.01 * N_RAW:
    raise ValueError(
        f"{N_INVALID:,} of {N_RAW:,} timestamps failed to parse with format '{TIMESTAMP_FORMAT}'. "
        "Inspect the raw Timestamp strings and adjust TIMESTAMP_FORMAT."
    )

# Explicit handling: invalid rows are excluded (reported, never silent)
EXCLUDED_TRANSACTION_IDS = np.array([], dtype=np.int64)
if N_INVALID > 0:
    EXCLUDED_TRANSACTION_IDS = df.loc[invalid_mask, "Transaction ID"].to_numpy()
    print("Excluded Transaction IDs (first 20):", EXCLUDED_TRANSACTION_IDS[:20])
    df = df.loc[~invalid_mask].reset_index(drop=True)
    print(f"Rows excluded from temporal features: {N_INVALID:,}")
    print("Note: Transaction IDs keep their ORIGINAL raw-file numbering, so gaps will exist.")

N_VALID = len(df)
print(f"Valid rows        : {N_VALID:,}")
print(f"Minimum timestamp : {df['Timestamp'].min()}")
print(f"Maximum timestamp : {df['Timestamp'].max()}")

del invalid_mask
gc.collect()

Invalid timestamps: 0
Valid rows        : 6,924,049
Minimum timestamp : 2022-09-01 00:00:00
Maximum timestamp : 2022-09-17 15:28:00


0

In [9]:
# Window lookups need each sender's transactions to be contiguous and in time order.
# Sort key: (Sender Account ID, Timestamp, Transaction ID).
# The Transaction ID tie-breaker makes the order deterministic when timestamps are equal
# (timestamps only have minute resolution).
# The DataFrame itself is NOT reordered. Only NumPy index arrays are built, to save memory.

sender_codes = df["Sender Account ID"].cat.codes.to_numpy().astype(np.int64)
ts_sec = df["Timestamp"].to_numpy().astype("datetime64[s]").astype(np.int64)   # epoch seconds
amount_paid = df["Amount Paid"].to_numpy(dtype=np.float64, copy=True)

n_missing_amount = int(np.isnan(amount_paid).sum())
if n_missing_amount:
    print(f"WARNING: {n_missing_amount:,} missing Amount Paid values treated as 0 in the 24h amount sum.")
    amount_paid = np.nan_to_num(amount_paid, nan=0.0)

tid = df["Transaction ID"].to_numpy()

t_start = time.time()
# np.lexsort: the LAST key is the primary key
order = np.lexsort((tid, ts_sec, sender_codes))

s = sender_codes[order]      # sorted sender codes
t = ts_sec[order]            # sorted timestamps (seconds)
a = amount_paid[order]       # sorted Amount Paid

assert (np.diff(s) >= 0).all(), "Sender codes are not grouped after sorting"
same_sender = s[1:] == s[:-1]
assert (np.diff(t)[same_sender] >= 0).all(), "Timestamps not ascending within sender"
print(f"Sorted {N_VALID:,} rows in {time.time() - t_start:,.1f} s")

Sorted 6,924,049 rows in 1.3 s


In [10]:
t_start = time.time()
n = len(s)

# ---------------------------------------------------------------
# COMPOSITE KEY: one globally sorted array, so searchsorted works for ALL senders at once.
#   key = sender_code * M + (timestamp - t_min + WINDOW_7D)
# The WINDOW_7D offset keeps (key - window) inside the sender's own block, so a window can
# never leak into the previous sender's rows. M is wider than any block.
# ---------------------------------------------------------------
t_min = int(t.min())
t_span = int(t.max()) - t_min
M = t_span + 2 * WINDOW_7D + 10
assert (int(s.max()) + 1) * M < 2**62, "Composite key would overflow int64"

key = s * M + (t - t_min + WINDOW_7D)          # sorted ascending
print(f"[1/5] composite key built        ({time.time() - t_start:,.1f} s)")

# ---------------------------------------------------------------
# RIGHT boundary: first index whose key >= current key (side="left").
# This is the first row of the current timestamp, so rows at the same timestamp
# (including the current row) fall at or after it and are NOT counted.
# ---------------------------------------------------------------
right = np.searchsorted(key, key, side="left")
print(f"[2/5] right boundary found       ({time.time() - t_start:,.1f} s)")

# ---------------------------------------------------------------
# LEFT boundary for window W: first index whose timestamp is strictly greater than (t - W)
# (side="right"). Window = (t - W, t), open on both ends.
# count = right - left
# ---------------------------------------------------------------
def count_in_window(window_seconds):
    left = np.searchsorted(key, key - window_seconds, side="right")
    return left, (right - left)

left_1h, cnt_1h = count_in_window(WINDOW_1H)
left_24h, cnt_24h = count_in_window(WINDOW_24H)
left_7d, cnt_7d = count_in_window(WINDOW_7D)
print(f"[3/5] window counts computed     ({time.time() - t_start:,.1f} s)")

# ---------------------------------------------------------------
# Amount Last 24 Hours: per-sender EXCLUSIVE prefix sum.
#   prefix[j] = sum of Amount Paid of the sender's rows strictly before row j
# so the sum over rows [left, right) = prefix[right] - prefix[left].
# Row `right` is not included, so the current amount is never added.
# The per-sender cumsum keeps float error relative to one sender's total, not the global total.
# ---------------------------------------------------------------
cum_incl = pd.Series(a).groupby(s, sort=False).cumsum().to_numpy()
prefix = cum_incl - a
amt_24h = prefix[right] - prefix[left_24h]
amt_24h = np.maximum(amt_24h, 0.0)             # remove tiny negative float noise
del cum_incl, prefix
print(f"[4/5] amount window computed     ({time.time() - t_start:,.1f} s)")

# ---------------------------------------------------------------
# Time Since Previous Transaction (seconds): difference to the previous row
# in the sorted order, only if that row belongs to the same sender. First row per sender = NaN.
# ---------------------------------------------------------------
time_since_prev = np.full(n, np.nan, dtype=np.float64)
idx = np.flatnonzero(s[1:] == s[:-1]) + 1
time_since_prev[idx] = (t[idx] - t[idx - 1]).astype(np.float64)
del idx
print(f"[5/5] time-since-previous done   ({time.time() - t_start:,.1f} s)")

# ---------------------------------------------------------------
# Restore the original row order (ascending Transaction ID)
# ---------------------------------------------------------------
def restore_order(x_sorted):
    out = np.empty_like(x_sorted)
    out[order] = x_sorted
    return out

res_1h = restore_order(cnt_1h).astype(np.int32)
res_24h = restore_order(cnt_24h).astype(np.int32)
res_7d = restore_order(cnt_7d).astype(np.int32)
res_amt = restore_order(amt_24h)
res_tsp = restore_order(time_since_prev)

# Free the large temporaries
del key, right, left_1h, left_24h, left_7d, cnt_1h, cnt_24h, cnt_7d, amt_24h, time_since_prev
del s, t, a, sender_codes, ts_sec, amount_paid, order
gc.collect()

print(f"Temporal features computed in {time.time() - t_start:,.1f} s total")

[1/5] composite key built        (0.1 s)
[2/5] right boundary found       (0.4 s)
[3/5] window counts computed     (1.4 s)
[4/5] amount window computed     (1.8 s)
[5/5] time-since-previous done   (2.0 s)
Temporal features computed in 2.5 s total


In [11]:
temporal_features = pd.DataFrame({
    "Transaction ID": df["Transaction ID"].to_numpy(),
    "Sender Account ID": df["Sender Account ID"],
    "Receiver Account ID": df["Receiver Account ID"],
    "Transactions Last 1 Hour": res_1h,
    "Transactions Last 24 Hours": res_24h,
    "Transactions Last 7 Days": res_7d,
    "Amount Last 24 Hours": res_amt,
    "Time Since Previous Transaction": res_tsp,    # seconds, NaN for a sender's first transaction
})[FINAL_COLUMNS]

# Rows are already in raw-file order; make sure
assert temporal_features["Transaction ID"].is_monotonic_increasing, "Transaction ID order broken"

# Debug-only series (NOT part of the output), aligned by index
ts_debug = df["Timestamp"]
amt_debug = df["Amount Paid"].fillna(0.0)

del res_1h, res_24h, res_7d, res_amt, res_tsp
gc.collect()

print(f"temporal_features shape: {temporal_features.shape}")
print(f"Memory usage: {temporal_features.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")

temporal_features shape: (6924049, 8)
Memory usage: 368.9 MB


In [13]:
import time

RUN_ACCOUNTS_COVERAGE = False   # set True only if you want the (slower) accounts.csv check
N_SAMPLE_ACCOUNTS = 25

def tick(label, t0):
    print(f"   [{time.time() - t0:5.1f} s] {label}")

T0 = time.time()
print("=" * 70)
print("VALIDATION")
print("=" * 70)

tf = temporal_features   # alias, no copy

# --- Row count -------------------------------------------------------
print(f"Raw rows                : {N_RAW:,}")
print(f"Invalid timestamps      : {N_INVALID:,} (excluded)")
print(f"Valid rows              : {N_VALID:,}")
print(f"temporal_features rows  : {len(tf):,}")
assert len(tf) == N_VALID
assert N_VALID + N_INVALID == N_RAW
print("OK: rows == valid transaction rows")

# --- Transaction ID --------------------------------------------------
tid_arr = tf["Transaction ID"].to_numpy()
assert (np.diff(tid_arr) > 0).all()          # strictly increasing => unique and ordered
print("OK: Transaction ID is unique and ascending")
if N_INVALID == 0:
    assert tid_arr[0] == 1 and tid_arr[-1] == N_RAW
    print(f"OK: Transaction ID covers 1..{N_RAW:,}")

# --- Schema ----------------------------------------------------------
assert list(tf.columns) == FINAL_COLUMNS
print("OK: exactly the 8 required columns, in the required order")
tick("schema / IDs", T0)

# --- Pull numeric columns once as NumPy arrays (fast) -----------------
c1 = tf["Transactions Last 1 Hour"].to_numpy()
c24 = tf["Transactions Last 24 Hours"].to_numpy()
c7 = tf["Transactions Last 7 Days"].to_numpy()
amt = tf["Amount Last 24 Hours"].to_numpy()
tsp = tf["Time Since Previous Transaction"].to_numpy()
sender_codes_f = tf["Sender Account ID"].cat.codes.to_numpy()

# --- Missing values --------------------------------------------------
n_nan_tsp = int(np.isnan(tsp).sum())
n_unique_senders = int(np.unique(sender_codes_f).size)
print("\nMissing values per column:")
print(tf.isna().sum().to_string())
print(f"\nUnique senders: {n_unique_senders:,} | NaN in Time Since Previous: {n_nan_tsp:,}")
assert n_nan_tsp == n_unique_senders, "NaN should appear exactly once per sender"
assert not tf[FINAL_COLUMNS[:7]].isna().any().any()
print("OK: NaN only in Time Since Previous Transaction, exactly once per sender")
tick("missing values", T0)

# --- Numeric validity ------------------------------------------------
assert (c1 >= 0).all() and (c24 >= 0).all() and (c7 >= 0).all() and (amt >= 0).all()
assert (tsp[~np.isnan(tsp)] >= 0).all()
print("OK: all counts / amounts / times are >= 0")

# --- Logical checks --------------------------------------------------
assert (c1 <= c24).all() and (c24 <= c7).all()
print("OK: Last 1 Hour <= Last 24 Hours <= Last 7 Days for every row")
assert (amt[c24 == 0] == 0).all()
print("OK: zero 24h count -> zero 24h amount")
tick("numeric / logical checks", T0)

# --- Independent brute-force cross-check on sampled accounts ----------
# Works on integer category codes only: no string comparisons on 6.9M rows.
rng = np.random.default_rng(42)
counts = np.bincount(sender_codes_f)
cand = np.flatnonzero((counts >= 30) & (counts <= 400))
if cand.size == 0:
    cand = np.argsort(counts)[::-1][:N_SAMPLE_ACCOUNTS]
picked = rng.choice(cand, size=min(N_SAMPLE_ACCOUNTS, cand.size), replace=False)

# One pass: row indices of the picked senders, grouped by sender via a stable sort of the sample only
rows = np.flatnonzero(np.isin(sender_codes_f, picked))
rows = rows[np.argsort(sender_codes_f[rows], kind="stable")]
ts_all = ts_debug.to_numpy().astype("datetime64[s]").astype(np.int64)[rows] if False else None
ts_rows = ts_debug.to_numpy()[rows].astype("datetime64[s]").astype(np.int64)
amt_rows = amt_debug.to_numpy()[rows]
code_rows = sender_codes_f[rows]
tid_rows = tid_arr[rows]

bounds = np.flatnonzero(np.diff(code_rows)) + 1
for sl in np.split(np.arange(rows.size), bounds):
    order_g = np.lexsort((tid_rows[sl], ts_rows[sl]))     # (timestamp, Transaction ID)
    idx = rows[sl][order_g]
    tt = ts_rows[sl][order_g]
    aa = amt_rows[sl][order_g]
    diff = tt[:, None] - tt[None, :]                      # t_i - t_j

    def brute_count(W):
        return ((diff > 0) & (diff < W)).sum(axis=1)      # t - W < t_j < t

    exp_amt = (((diff > 0) & (diff < WINDOW_24H)) * aa[None, :]).sum(axis=1)
    exp_tsp = np.concatenate([[np.nan], np.diff(tt).astype(float)])

    assert np.array_equal(c1[idx], brute_count(WINDOW_1H))
    assert np.array_equal(c24[idx], brute_count(WINDOW_24H))
    assert np.array_equal(c7[idx], brute_count(WINDOW_7D))
    assert np.allclose(amt[idx], exp_amt, rtol=1e-9, atol=1e-6)
    assert np.allclose(tsp[idx], exp_tsp, equal_nan=True)

print(f"OK: brute-force check matched exactly on {picked.size} sampled sender accounts "
      f"({rows.size:,} transactions)")
tick("brute-force cross-check", T0)

# --- Optional: coverage against LI-Small_accounts.csv (slow, off by default)
if RUN_ACCOUNTS_COVERAGE:
    acc = pd.read_csv(ACCOUNTS_FILE, usecols=["Bank ID", "Account Number"], dtype=str)
    acc_ids = pd.Index(acc["Bank ID"] + "_" + acc["Account Number"])
    cov = tf["Sender Account ID"].cat.categories.isin(acc_ids).mean() * 100   # hash-based, fast
    print(f"Info: {cov:.2f}% of distinct sender IDs are present in LI-Small_accounts.csv")
    del acc, acc_ids
    tick("accounts coverage", T0)

del c1, c24, c7, amt, tsp, sender_codes_f, tid_arr, rows, ts_rows, amt_rows, code_rows, tid_rows
gc.collect()
print(f"\nALL VALIDATION CHECKS PASSED  (total {time.time() - T0:,.1f} s)")

VALIDATION
Raw rows                : 6,924,049
Invalid timestamps      : 0 (excluded)
Valid rows              : 6,924,049
temporal_features rows  : 6,924,049
OK: rows == valid transaction rows
OK: Transaction ID is unique and ascending
OK: Transaction ID covers 1..6,924,049
OK: exactly the 8 required columns, in the required order
   [  0.0 s] schema / IDs

Missing values per column:
Transaction ID                          0
Sender Account ID                       0
Receiver Account ID                     0
Transactions Last 1 Hour                0
Transactions Last 24 Hours              0
Transactions Last 7 Days                0
Amount Last 24 Hours                    0
Time Since Previous Transaction    681283

Unique senders: 681,283 | NaN in Time Since Previous: 681,283
OK: NaN only in Time Since Previous Transaction, exactly once per sender
   [  0.6 s] missing values
OK: all counts / amounts / times are >= 0
OK: Last 1 Hour <= Last 24 Hours <= Last 7 Days for every row
OK: zero 

In [14]:
from IPython.display import display

print("First 20 rows of temporal_features:")
display(temporal_features.head(20))

# Manual-inspection view: sender accounts with several transactions
cnts = temporal_features["Sender Account ID"].value_counts()
sample_accounts = list(cnts[(cnts >= 10) & (cnts <= 60)].index[:3]) or list(cnts.index[:3])

DEBUG_COLS = [
    "Transaction ID", "Sender Account ID", "Timestamp",
    "Transactions Last 1 Hour", "Transactions Last 24 Hours", "Transactions Last 7 Days",
    "Amount Last 24 Hours", "Time Since Previous Transaction",
]

for acct in sample_accounts:
    m = (temporal_features["Sender Account ID"] == acct).to_numpy()
    dbg = temporal_features.loc[m].copy()
    dbg["Timestamp"] = ts_debug[m].to_numpy()          # debug only, not in final output
    dbg["Amount Paid (current)"] = amt_debug[m].to_numpy()
    dbg = dbg.sort_values(["Timestamp", "Transaction ID"])[DEBUG_COLS + ["Amount Paid (current)"]]
    print(f"\nSender account: {acct}  ({len(dbg)} transactions)")
    display(dbg.head(15))

First 20 rows of temporal_features:


,Transaction ID,Sender Account ID,Receiver Account ID,Transactions Last 1 Hour,Transactions Last 24 Hours,Transactions Last 7 Days,Amount Last 24 Hours,Time Since Previous Transaction
0,1,011_8000ECA90,011_8000ECA90,2,2,2,592593.97,60.0
1,2,03402_80021DAD0,03402_80021DAD0,0,0,0,0.00,NaN
2,3,011_8000ECA90,001120_8006AA910,0,0,0,0.00,NaN
3,4,03814_8006AD080,03814_8006AD080,0,0,0,0.00,NaN
4,5,020_8006AD530,020_8006AD530,0,0,0,0.00,NaN
5,6,012_8006ADD30,012_8006ADD30,2,2,2,2208375.71,1440.0
6,7,011_800059120,01217_8006AD4E0,0,0,0,0.00,NaN
7,8,011_8000ECA90,011_8000ECA90,1,1,1,592571.00,420.0
8,9,001120_8006AA910,0243166_81470DCF0,0,0,0,0.00,NaN
9,10,01217_8006AD4E0,01217_8006AD4E0,0,0,0,0.00,NaN



Sender account: 000_8001263A0  (60 transactions)


,Transaction ID,Sender Account ID,Timestamp,Transactions Last 1 Hour,Transactions Last 24 Hours,Transactions Last 7 Days,Amount Last 24 Hours,Time Since Previous Transaction,Amount Paid (current)
918892,918893,000_8001263A0,2022-09-01 10:30:00,0,0,0,0.00,NaN,8036.08
918893,918894,000_8001263A0,2022-09-01 10:42:00,1,1,1,8036.08,720.0,37977.25
918891,918892,000_8001263A0,2022-09-01 10:52:00,2,2,2,46013.33,600.0,67531.01
1190974,1190975,000_8001263A0,2022-09-01 16:32:00,0,3,3,113544.34,20400.0,458.41
1190975,1190976,000_8001263A0,2022-09-01 16:41:00,1,4,4,114002.75,540.0,481.55
1487256,1487257,000_8001263A0,2022-09-01 23:03:00,0,5,5,114484.30,22920.0,1121.20
1487255,1487256,000_8001263A0,2022-09-01 23:17:00,1,6,6,115605.50,840.0,1140.46
1678199,1678200,000_8001263A0,2022-09-02 03:02:00,0,7,7,116745.96,13500.0,41918.27
1678200,1678201,000_8001263A0,2022-09-02 03:10:00,1,8,8,158664.23,480.0,10309.78
1869375,1869376,000_8001263A0,2022-09-02 07:34:00,0,9,9,168974.01,15840.0,458.41



Sender account: 000_800274EB0  (60 transactions)


,Transaction ID,Sender Account ID,Timestamp,Transactions Last 1 Hour,Transactions Last 24 Hours,Transactions Last 7 Days,Amount Last 24 Hours,Time Since Previous Transaction,Amount Paid (current)
166444,166445,000_800274EB0,2022-09-01 00:06:00,0,0,0,0.00,NaN,41865.20
983328,983329,000_800274EB0,2022-09-01 12:14:00,0,1,1,41865.20,43680.0,13884.16
983329,983330,000_800274EB0,2022-09-01 12:18:00,1,2,2,55749.36,240.0,366349.43
983327,983328,000_800274EB0,2022-09-01 12:27:00,2,3,3,422098.79,540.0,95356.47
1258050,1258051,000_800274EB0,2022-09-01 18:02:00,0,4,4,517455.26,20100.0,69.00
1258048,1258049,000_800274EB0,2022-09-01 18:04:00,1,5,5,517524.26,120.0,1511.90
1258051,1258052,000_800274EB0,2022-09-01 18:06:00,2,6,6,519036.16,120.0,87.13
1258049,1258050,000_800274EB0,2022-09-01 18:14:00,3,7,7,519123.29,480.0,1931.27
1867878,1867879,000_800274EB0,2022-09-02 07:33:00,0,7,8,479189.36,47940.0,1931.27
1867879,1867880,000_800274EB0,2022-09-02 07:34:00,1,8,9,481120.63,60.0,69.00



Sender account: 000_80084E3C0  (60 transactions)


,Transaction ID,Sender Account ID,Timestamp,Transactions Last 1 Hour,Transactions Last 24 Hours,Transactions Last 7 Days,Amount Last 24 Hours,Time Since Previous Transaction,Amount Paid (current)
174442,174443,000_80084E3C0,2022-09-01 00:00:00,0,0,0,0.00,NaN,352.77
174441,174442,000_80084E3C0,2022-09-01 00:17:00,1,1,1,352.77,1020.0,393.75
460995,460996,000_80084E3C0,2022-09-01 00:31:00,2,2,2,746.52,840.0,751.01
460994,460995,000_80084E3C0,2022-09-01 00:32:00,3,3,3,1497.53,60.0,904.85
594012,594013,000_80084E3C0,2022-09-01 03:34:00,0,4,4,2402.38,10920.0,268.00
594010,594011,000_80084E3C0,2022-09-01 03:40:00,1,5,5,2670.38,360.0,903.06
594011,594012,000_80084E3C0,2022-09-01 03:55:00,2,6,6,3573.44,900.0,192.18
1762782,1762783,000_80084E3C0,2022-09-02 05:05:00,0,0,7,0.00,90600.0,393.75
1762527,1762528,000_80084E3C0,2022-09-02 05:06:00,1,1,8,393.75,60.0,903.06
1762528,1762529,000_80084E3C0,2022-09-02 05:11:00,2,2,9,1296.81,300.0,192.18


In [15]:
t_start = time.time()
temporal_features.to_csv(OUTPUT_FILE, index=False)

print("Saved successfully:")
print(os.path.abspath(OUTPUT_FILE))
print("\nFinal shape:")
print(temporal_features.shape)
print(f"File size: {os.path.getsize(OUTPUT_FILE) / 1024**2:,.1f} MB | write time: {time.time() - t_start:,.1f} s")

# Quick read-back sanity check
check = pd.read_csv(OUTPUT_FILE, nrows=5)
assert list(check.columns) == FINAL_COLUMNS
display(check)

Saved successfully:
e:\AML-GNN-Project\data\processed\temporal_features.csv

Final shape:
(6924049, 8)
File size: 465.4 MB | write time: 37.9 s


,Transaction ID,Sender Account ID,Receiver Account ID,Transactions Last 1 Hour,Transactions Last 24 Hours,Transactions Last 7 Days,Amount Last 24 Hours,Time Since Previous Transaction
0,1,011_8000ECA90,011_8000ECA90,2,2,2,592593.97,60.0
1,2,03402_80021DAD0,03402_80021DAD0,0,0,0,0.00,NaN
2,3,011_8000ECA90,001120_8006AA910,0,0,0,0.00,NaN
3,4,03814_8006AD080,03814_8006AD080,0,0,0,0.00,NaN
4,5,020_8006AD530,020_8006AD530,0,0,0,0.00,NaN
